<a href="https://colab.research.google.com/github/wyldescience/FolSum/blob/main/Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf /content/drive


from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
ROOT = "ROOT DIRECTORY"
VAL_DIR = os.path.join(ROOT, "WHERE TO SAVE VALIDATION RESULTS")
IN_FULL = os.path.join(VAL_DIR, "UNCROPPED IMAGES")

print("Exists ROOT?", os.path.exists(ROOT), ROOT)
print("Exists validation?", os.path.exists(VAL_DIR), VAL_DIR)
print("Exists images_full?", os.path.exists(IN_FULL), IN_FULL)

# list what's inside validation + images_full
print("\nValidation contents:", os.listdir(VAL_DIR) if os.path.exists(VAL_DIR) else "MISSING")
print("\nimages_full contents:", os.listdir(IN_FULL)[:20] if os.path.exists(IN_FULL) else "MISSING")


In [ ]:
# ====== CHUNK 1: Crop + Tile validation images ======

import os, glob, math
import cv2
import numpy as np

# ---------------- Paths ----------------
OUT_CROP = os.path.join(VAL_DIR, "images_cropped")
OUT_TILES_IMG = os.path.join(VAL_DIR, "tiles", "images")

os.makedirs(OUT_CROP, exist_ok=True)
os.makedirs(OUT_TILES_IMG, exist_ok=True)

# ---------------- Arena crop (circle-fit) ----------------
def crop_arena_by_contour(img_rgb, pad=15):
    if img_rgb.ndim == 2:
        img_rgb = np.stack([img_rgb]*3, axis=-1)
    if img_rgb.shape[-1] == 4:
        img_rgb = img_rgb[..., :3]
    img_rgb = img_rgb.astype(np.uint8)

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k1)
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return img_rgb

    H, W = gray.shape
    cx0, cy0 = W / 2, H / 2

    best, best_score = None, -1e18
    for c in cnts:
        area = cv2.contourArea(c)
        if area < 0.05 * H * W:
            continue
        peri = cv2.arcLength(c, True)
        if peri <= 0:
            continue
        circularity = 4 * np.pi * area / (peri * peri)
        (x, y), r = cv2.minEnclosingCircle(c)
        dist = ((x - cx0) ** 2 + (y - cy0) ** 2) ** 0.5
        score = (circularity * 2.0) + (area / (H * W)) - (dist / max(H, W))
        if score > best_score:
            best_score, best = score, c

    if best is None:
        best = max(cnts, key=cv2.contourArea)

    (x, y), r = cv2.minEnclosingCircle(best)
    x, y, r = int(x), int(y), int(r)

    arena_mask = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(arena_mask, (x, y), r, 255, thickness=-1)

    x0 = max(0, x - r - pad); x1 = min(W, x + r + pad)
    y0 = max(0, y - r - pad); y1 = min(H, y + r + pad)

    cropped = img_rgb[y0:y1, x0:x1].copy()
    m = arena_mask[y0:y1, x0:x1] > 0
    cropped[~m] = 0
    return cropped

# ---------------- Tiling ----------------
def tile_image(img_rgb, tile_size=640, overlap=0.25):
    """
    Returns list of (tile_rgb, x0, y0) in the CROPPED image coordinate frame.
    """
    H, W = img_rgb.shape[:2]
    stride = int(tile_size * (1 - overlap))
    stride = max(1, stride)

    tiles = []
    # Ensure coverage to edges
    xs = list(range(0, max(1, W - tile_size + 1), stride))
    ys = list(range(0, max(1, H - tile_size + 1), stride))
    if len(xs) == 0: xs = [0]
    if len(ys) == 0: ys = [0]
    if xs[-1] != max(0, W - tile_size): xs.append(max(0, W - tile_size))
    if ys[-1] != max(0, H - tile_size): ys.append(max(0, H - tile_size))

    for y0 in ys:
        for x0 in xs:
            tile = img_rgb[y0:y0+tile_size, x0:x0+tile_size].copy()
            # Pad if needed (for small images)
            th, tw = tile.shape[:2]
            if th < tile_size or tw < tile_size:
                pad_img = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                pad_img[:th, :tw] = tile
                tile = pad_img
            tiles.append((tile, x0, y0))
    return tiles

# ---------------- Run crop+tile on all validation images ----------------
tile_size = 640
overlap = 0.25   # 0.2–0.35 typical; more overlap = fewer misses but more duplicates to merge later
pad = 15

img_paths = sorted(glob.glob(os.path.join(IN_FULL, "*.jpg")) +
                   glob.glob(os.path.join(IN_FULL, "*.JPG")) +
                   glob.glob(os.path.join(IN_FULL, "*.png")))

print("Found validation full images:", len(img_paths))

# Optional: clear old tiles
# for f in glob.glob(os.path.join(OUT_TILES_IMG, "*")): os.remove(f)

for p in img_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    bgr = cv2.imread(p)
    if bgr is None:
        print("Skipping unreadable:", p)
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    cropped = crop_arena_by_contour(rgb, pad=pad)

    # save cropped
    crop_out = os.path.join(OUT_CROP, stem + "_crop.jpg")
    cv2.imwrite(crop_out, cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR))

    # tile cropped
    tiles = tile_image(cropped, tile_size=tile_size, overlap=overlap)

    for i, (tile, x0, y0) in enumerate(tiles):
        # filename encodes which original + tile origin in CROPPED space
        out_name = f"{stem}__x{x0}_y{y0}__ts{tile_size}.jpg"
        out_path = os.path.join(OUT_TILES_IMG, out_name)
        cv2.imwrite(out_path, cv2.cvtColor(tile, cv2.COLOR_RGB2BGR))

print("Done. Crops in:", OUT_CROP)
print("Tiles in:", OUT_TILES_IMG)


In [ ]:
# ====== CHUNK 2: Predict on tiles + recombine with NMS + count per original ======
!pip -q install ultralytics opencv-python

import os, glob, re
import cv2
import numpy as np
import pandas as pd

from ultralytics import YOLO

# ---------------- Paths ----------------
ROOT = "/content/drive/MyDrive/YOLO nymph detector"
VAL_DIR = os.path.join(ROOT, "validation")

CROPS_DIR = os.path.join(VAL_DIR, "images_cropped")
TILES_DIR = os.path.join(VAL_DIR, "tiles", "images")
OUT_OVER = os.path.join(VAL_DIR, "pred_overlays")
os.makedirs(OUT_OVER, exist_ok=True)

# CHANGE THIS to your trained best.pt:
# Example:
# BEST_WEIGHTS = "/content/drive/MyDrive/YOLO nymph detector/yolo_runs/nymph_yolov8n_tiles/weights/best.pt"
BEST_WEIGHTS = "WEIGHTS FROM BEST MODEL"

model = YOLO(BEST_WEIGHTS)

# ---------------- Utilities ----------------
tile_pat = re.compile(r"^(?P<stem>.+)__x(?P<x>\d+)_y(?P<y>\d+)__ts(?P<ts>\d+)\.jpg$", re.IGNORECASE)

def iou_xyxy(a, b):
    # a,b: [x1,y1,x2,y2]
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter <= 0: return 0.0
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def nms_numpy(boxes, scores, iou_thr=0.5):
    """
    boxes: Nx4, scores: N
    Returns indices kept.
    """
    if len(boxes) == 0:
        return []
    idxs = np.argsort(scores)[::-1].tolist()
    keep = []
    while idxs:
        i = idxs.pop(0)
        keep.append(i)
        idxs = [j for j in idxs if iou_xyxy(boxes[i], boxes[j]) < iou_thr]
    return keep

def draw_boxes(img_rgb, boxes, color=(0,255,0), thickness=2):
    out = img_rgb.copy()
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(out, (int(x1),int(y1)), (int(x2),int(y2)), color, thickness)
    return out

# ---------------- Settings ----------------
conf = 0.35        # increase to reduce false positives (e.g., 0.35 or 0.5)
iou_tile = 0.6     # NMS within tile is handled by YOLO; this affects internal YOLO NMS
iou_global = 0.5   # NMS across tiles to merge duplicates from overlap
max_det = 3000     # allow lots of detections if images are dense
imgsz = 640        # must match tile size

# ---------------- Group tiles by original stem ----------------
tile_paths = sorted(glob.glob(os.path.join(TILES_DIR, "*.jpg")))
by_stem = {}
for tp in tile_paths:
    fn = os.path.basename(tp)
    m = tile_pat.match(fn)
    if not m:
        continue
    stem = m.group("stem")
    x0 = int(m.group("x")); y0 = int(m.group("y"))
    ts = int(m.group("ts"))
    by_stem.setdefault(stem, []).append((tp, x0, y0, ts))

print("Found tile groups:", len(by_stem))

# ---------------- Run inference + recombine ----------------
rows = []

for stem, tiles in by_stem.items():
    # load cropped image for QC overlay
    crop_path = os.path.join(CROPS_DIR, stem + "_crop.jpg")
    crop_bgr = cv2.imread(crop_path)
    if crop_bgr is None:
        print("Missing cropped image for:", stem, "expected:", crop_path)
        continue
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    all_boxes = []
    all_scores = []

    # predict tile-by-tile (simple + reliable)
    for (tp, x0, y0, ts) in tiles:
        res = model.predict(
            source=tp,
            imgsz=imgsz,
            conf=conf,
            iou=iou_tile,
            max_det=max_det,
            verbose=False
        )[0]

        if res.boxes is None or len(res.boxes) == 0:
            continue

        # tile-space boxes -> cropped-image boxes
        xyxy = res.boxes.xyxy.cpu().numpy()
        scores = res.boxes.conf.cpu().numpy()

        for (b, s) in zip(xyxy, scores):
            x1, y1, x2, y2 = b
            # shift by tile origin
            all_boxes.append([x1 + x0, y1 + y0, x2 + x0, y2 + y0])
            all_scores.append(float(s))

    all_boxes = np.array(all_boxes, dtype=float)
    all_scores = np.array(all_scores, dtype=float)

    if len(all_boxes) == 0:
        final_boxes = np.zeros((0,4), dtype=float)
    else:
        keep = nms_numpy(all_boxes, all_scores, iou_thr=iou_global)
        final_boxes = all_boxes[keep]

    count = int(len(final_boxes))

    # save overlay
    overlay = draw_boxes(crop_rgb, final_boxes, color=(0,255,0), thickness=2)
    out_path = os.path.join(OUT_OVER, stem + "_overlay.jpg")
    cv2.imwrite(out_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    rows.append({
        "image_stem": stem,
        "pred_count": count,
        "n_tiles": len(tiles),
        "conf": conf,
        "iou_global": iou_global,
        "overlay_path": out_path
    })

df = pd.DataFrame(rows).sort_values("image_stem")
out_csv = os.path.join(VAL_DIR, "counts.csv")
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
df.head(10)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

df = pd.read_csv("RESULTS OF VALIDATION CSV")

auto = df["auto"].to_numpy()
manual = df["manual"].to_numpy()
error = auto - manual

r, p = pearsonr(auto, manual)

mae = np.mean(np.abs(error))
bias = np.mean(error)

# Percent errors
ape = np.abs(error) / manual * 100
mape = np.mean(ape)
mdape = np.median(ape)

# Robust alternatives
smape = np.mean(200 * np.abs(error) / (np.abs(auto) + np.abs(manual)))  # symmetric MAPE, bounded
mape_over_50 = np.mean(ape[manual >= 50])
mape_over_100 = np.mean(ape[manual >= 100])

# Percentage of images with manual count <= 60
pct_manual_le_60 = np.mean(manual <= 60) * 100

print(f"Images with manual count ≤ 60: {pct_manual_le_60:.1f}%")

print(f"Pearson r = {r:.3f} (p = {p:.2e})")
print(f"MAE = {mae:.1f} individuals")
print(f"Bias = {bias:.1f} individuals")
print(f"MAPE = {mape:.1f}%")
print(f"Median APE (MdAPE) = {mdape:.1f}%")
print(f"sMAPE = {smape:.1f}%")
print(f"MAPE (manual>=50) = {mape_over_50:.1f}%")
print(f"MAPE (manual>=100) = {mape_over_100:.1f}%")


Model performance was evaluated on an independent validation dataset comprising 20 images not used during training or hyperparameter tuning. Automated counts were obtained by applying the trained detector to tiled image patches and recombining detections across tiles.

Automated and manual counts showed extremely strong agreement (Pearson r = 0.992, p = 1.35 × 10⁻¹⁷). The mean absolute error was 17.1 individuals, with a small positive bias of 15.1 individuals. While mean absolute percentage error (MAPE) was inflated by low-count images, median absolute percentage error was 10.1%. For images containing ≥50 individuals, MAPE was 11.0%, and for ≥100 individuals it was 9.8%, indicating high accuracy at biologically relevant densities.

Pearson r = 0.992 (p = 1.35e-17)
MAE = 17.1 individuals
Bias = 15.1 individuals
MAPE = 63.3%
Median APE (MdAPE) = 10.1%
sMAPE = 24.5%
MAPE (manual>=50) = 11.0%
MAPE (manual>=100) = 9.8%

Springtail nymphs were detected using a convolutional neural network object detector based on YOLOv8, implemented via the Ultralytics framework (Jocher et al., 2023). The model was trained using transfer learning from COCO-pretrained weights and applied to tiled image patches derived from cropped arena images. Training and inference were performed in Python using PyTorch (Paszke et al., 2019), with image preprocessing implemented in OpenCV. The trained model weights and inference pipeline are provided as supplementary material.

image tiles were 640 x 640 pixels:

Cropped arena images were subdivided into overlapping 640 × 640 pixel tiles prior to annotation, training, and inference. Overlapping tiles ensured that individuals located near tile boundaries were fully represented in at least one tile.
If you run the trained model on new data:

Use the same tile size (640)

Use the same overlap

Do not resize full images directly to 640

If you change tile size, you should revalidate.

**Regression model for count correction**

Given there is systematic bias in the model where it tends to overcount when images have low number of nymphs run a regression and see how much it differs from linear.

Automated counts were highly correlated with manual counts (R² = 0.984). However, automated estimates showed a small but systematic positive bias at low population densities. We therefore fitted a linear calibration model using an independent validation dataset (n = 20 containers), relating automated to manual counts. This calibration was applied to all automated estimates prior to analysis.

YOLO gives highly reliable relative counts across images, but slightly overestimates at low densities. We quantified this bias using an independent validation set and corrected it using a simple calibration model. After correction, automated counts closely approximate manual counts across the full range.


Quadratic model better
for Linear Corrected count = −24.93 + 1.06 × (automated count).

In [ ]:
import statsmodels.api as sm
import numpy as np

X_lin = sm.add_constant(auto)
model_lin = sm.OLS(manual, X_lin).fit()

print(model_lin.summary())



# Design matrix: intercept + auto + auto^2
X_quad = np.column_stack([auto, auto**2])
X_quad = sm.add_constant(X_quad)

model_quad = sm.OLS(manual, X_quad).fit()

print(model_quad.summary())

print(f"Linear model:    AIC = {model_lin.aic:.2f}, BIC = {model_lin.bic:.2f}")
print(f"Quadratic model: AIC = {model_quad.aic:.2f}, BIC = {model_quad.bic:.2f}")


f_test = model_quad.compare_f_test(model_lin)
print(f"F-test: F = {f_test[0]:.2f}, p = {f_test[1]:.3g}")


Apply regression with quadratic correction

> Add blockquote



In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("RESULTS OF VALIDATION CSV")

auto = df["auto"].to_numpy()
manual = df["manual"].to_numpy()

# Uncentred quadratic design matrix: [1, auto, auto^2]
X_quad = sm.add_constant(np.column_stack([auto, auto**2]))
model_quad = sm.OLS(manual, X_quad).fit()
print(model_quad.summary())

A0, A1, A2 = model_quad.params  # intercept, linear, quadratic

# Apply correction (predict manual from auto)
df["auto_corrected_quad"] = A0 + A1 * df["auto"] + A2 * (df["auto"] ** 2)
df["auto_corrected_quad"] = df["auto_corrected_quad"].clip(lower=0)

# Your linear correction
INTERCEPT = -20.58
SLOPE = 1.04
df["auto_corrected_linear"] = (INTERCEPT + SLOPE * df["auto"]).clip(lower=0)

# Compare
df["diff"] = df["auto_corrected_quad"] - df["auto_corrected_linear"]
print(df[["auto", "manual", "auto_corrected_quad", "auto_corrected_linear", "diff"]].head())

print(df.loc[df["manual"] <= 60, "diff"].describe())


PLot the nonlinear model correction with linear comparison

The quadratic correction improves fit primarily at low counts (≤60), where systematic underestimation is evident, while predictions at higher counts are largely unchanged.

Bias between automated and manual counts was corrected using ordinary least squares regression. In addition to a linear correction, we tested a quadratic model by including a squared automated-count term to account for systematic non-linearity at low counts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Smooth x-grid for plotting
x = np.linspace(df["auto"].min(), df["auto"].max(), 300)

# Linear correction (your existing model)
INTERCEPT = -20.58
SLOPE = 1.04
y_lin = INTERCEPT + SLOPE * x

# Quadratic correction (from uncentred fit)
# Replace A0, A1, A2 with your fitted coefficients if not already defined
y_quad = A0 + A1 * x + A2 * x**2

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

# ----- Linear: full range -----
axes[0].scatter(df["auto"], df["manual"], alpha=0.5)
axes[0].plot(x, y_lin, linewidth=2)
axes[0].set_title("Linear (full range)")
axes[0].set_xlabel("YOLO count")
axes[0].set_ylabel("Manual count")

# ----- Quadratic: full range -----
axes[1].scatter(df["auto"], df["manual"], alpha=0.5)
axes[1].plot(x, y_quad, linestyle="--", linewidth=2)
axes[1].set_title("Quadratic (full range)")
axes[1].set_xlabel("YOLO count")

# ----- Zoomed low-count region -----
axes[2].scatter(df["auto"], df["manual"], alpha=0.6)
axes[2].plot(x, y_lin, linewidth=2, label="Linear")
axes[2].plot(x, y_quad, linestyle="--", linewidth=2, label="Quadratic")
axes[2].set_xlim(0, 100)
axes[2].set_ylim(0, 120)
axes[2].set_title("Zoom: YOLO ≤ 100")
axes[2].set_xlabel("YOLO count")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# load validation data
df = pd.read_csv("RESULTS OF VALIDATION CSV")

# regression coefficients from your OLS
INTERCEPT = -20.58
SLOPE = 1.04

# apply correction
df["auto_corrected"] = INTERCEPT + SLOPE * df["auto"]

# enforce biological constraint
df["auto_corrected"] = df["auto_corrected"].clip(lower=0)

df.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(df["manual"], df["auto"], alpha=0.8)
plt.plot([0, df["manual"].max()], [0, df["manual"].max()], "k--", label="1:1 line")

plt.xlabel("Manual count")
plt.ylabel("YOLO count (raw)")
plt.title("Raw automated counts vs manual")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(df["manual"], df["auto_corrected"], alpha=0.8)
plt.plot([0, df["manual"].max()], [0, df["manual"].max()], "k--", label="1:1 line")

plt.xlabel("Manual count")
plt.ylabel("YOLO count (bias-corrected)")
plt.title("Bias-corrected automated counts vs manual")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import pearsonr

def metrics(pred, true):
    err = pred - true
    return {
        "r": pearsonr(pred, true)[0],
        "MAE": np.mean(np.abs(err)),
        "Bias": np.mean(err),
        "MdAPE": np.median(np.abs(err) / true * 100)
    }

print("Raw:")
print(metrics(df["auto"], df["manual"]))

print("\nCorrected:")
print(metrics(df["auto_corrected"], df["manual"]))


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr

# Load validation data
df = pd.read_csv("RESULTS OF VALIDATION CSV")

auto = df["auto"].values
manual = df["manual"].values

# OLS: manual ~ auto
X = sm.add_constant(auto)
ols = sm.OLS(manual, X).fit()


ols_table = pd.DataFrame({
    "Coefficient": ["Intercept", "Slope"],
    "Estimate": ols.params,
    "Std. Error": ols.bse,
    "t value": ols.tvalues,
    "p value": ols.pvalues,
    "95% CI lower": ols.conf_int()[0],
    "95% CI upper": ols.conf_int()[1],
})

ols_table

df["auto_corrected"] = ols.predict(X)


def metrics(pred, truth):
    error = pred - truth
    ape = np.abs(error) / truth * 100

    return {
        "Pearson r": pearsonr(pred, truth)[0],
        "MAE": np.mean(np.abs(error)),
        "Bias": np.mean(error),
        "Median APE (%)": np.median(ape),
        "sMAPE (%)": np.mean(200 * np.abs(error) / (np.abs(pred) + np.abs(truth)))
    }

raw_metrics = metrics(df["auto"].values, df["manual"].values)
corr_metrics = metrics(df["auto_corrected"].values, df["manual"].values)

summary_table = pd.DataFrame([raw_metrics, corr_metrics],
                             index=["Raw YOLO", "Bias-corrected YOLO"])

summary_table




In [ ]:
print("OLS calibration model:")
print(f"Manual count = {ols.params[0]:.2f} + {ols.params[1]:.3f} × Automated count")
print(f"R² = {ols.rsquared:.3f}, n = {len(df)}\n")

print("Performance comparison:")
print(summary_table.round(2).to_string())


In [ ]:
from statsmodels.iolib.summary2 import summary_col

summary = model.summary2()
summary
